# DT-HRES-S — Consolidated Model and Interface for Raspberry Pi

**EPICS in IEEE 2025-2026 — Open-source digital twin of a hybrid renewable energy system**

This notebook consolidates the trained model into a single artifact and defines the
interface that runs on the device. It is the bridge between the training environment
(Google Colab) and the field instrument (Raspberry Pi 5).

The notebook does three things:

1. Closes a pending task: a fair tuning of the neural network so the four-model
   comparison is honest, evaluated with leave-one-city-out.
2. Loads the winning model, validates it, and exports a single deployment bundle.
3. Defines a minimal, functional interface, with optional translation to an
   indigenous language of Yucatán, and runs it here as a preview of the device.

Sections 2 and 3 assume a model has been selected. Section 7 is the task that must
be settled first: until the neural network is properly tuned, the choice of model
is not yet defensible. The order in the notebook is deployment-first for reference,
but the tuning in section 7 is what unblocks the rest.

---

The 4D methodology (Concept, Body, Mind, Spirit) was used to organize the wider
project and is documented separately in the repository. It is offered there as a
development suggestion and is not repeated in this notebook, which stays focused on
deployment.


## 1. Setup

In [ ]:
import os, sys

if 'google.colab' in sys.modules and not os.path.exists('DT-HRES-S'):
    !git clone -q https://github.com/Technical-Arts-MTY/DT-HRES-S.git
    %cd DT-HRES-S
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from src.data_loader import load_city, load_all_cities, CITIES
from src import pv_model, wind_model, battery_model
from src import hres_simulator as hres
from src import ml_models

print('Setup complete')

## 2. Consolidate the model

The dataset is regenerated from the physics simulation across the four reference
cities, the selected model is trained on all available data, and the result is
checked before export. The model chosen here is the random forest, which offers
the best trade-off of accuracy, robustness and interpretability for this problem,
and runs on the Raspberry Pi CPU without an accelerator.


In [ ]:
config = hres.HRESConfig(
    pv=pv_model.PVSystem(p_rated_W=400, n_panels=10, tilt_deg=20, azimuth_deg=180),
    wind=wind_model.WindTurbine(rated_power_W=3000, rotor_diameter_m=4.0, hub_height_m=12.0),
    battery=battery_model.Battery(capacity_kWh=10, p_max_charge_kW=3.0, p_max_discharge_kW=3.0),
)

sims = []
for city in CITIES.keys():
    sims.append(hres.run(load_city(city), config))
df_all = pd.concat(sims, ignore_index=True)
print(f'Training set: {len(df_all):,} samples across {df_all["city"].nunique()} cities')

In [ ]:
from sklearn.ensemble import RandomForestRegressor

work = ml_models.cyclical_encode(df_all)
features = ml_models.DEFAULT_FEATURES + ['hour_sin', 'hour_cos', 'month_sin',
                                         'month_cos', 'doy_sin', 'doy_cos']
mask = work[features].dropna().index
X, y = work.loc[mask, features], work.loc[mask, 'p_pv_W']

model = RandomForestRegressor(
    n_estimators=200, max_depth=20, min_samples_leaf=5,
    n_jobs=-1, random_state=42,
)
model.fit(X, y)
print('Model trained')

### 2.1 Consistency check before export

A model that predicts physically impossible values must not reach the device.
The check below confirms predicted PV output stays within the plausible envelope
and reports the fraction of predictions that violate it.


In [ ]:
pred = model.predict(X)

# PV output cannot be negative, and should not exceed the array's rated power
# by a wide margin under real irradiance.
rated_W = config.pv.p_array_W
violations = ((pred < -1.0) | (pred > 1.15 * rated_W)).mean()

print(f'Rated array power: {rated_W/1000:.1f} kWp')
print(f'Predictions out of physical range: {violations*100:.2f} %')
if violations > 0.01:
    print('WARNING: review the model before deployment')
else:
    print('Consistency check passed')

## 3. Export the deployment bundle

A single file is copied to the Raspberry Pi. It contains the model, the exact
feature order it expects, and metadata for traceability. The device loads this
file and nothing else from the training pipeline.


In [ ]:
from datetime import datetime, timezone

bundle = {
    'model': model,
    'features': features,
    'target': 'p_pv_W',
    'config': {
        'pv_kWp': config.pv.p_array_W / 1000,
        'wind_kW': config.wind.rated_power_W / 1000,
        'battery_kWh': config.battery.capacity_kWh,
    },
    'trained_on': list(CITIES.keys()),
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'version': '1.0',
}

out_dir = Path('results/models')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'dt_hres_s_deployment.joblib'
joblib.dump(bundle, out_path, compress=3)

size_mb = out_path.stat().st_size / 1e6
print(f'Bundle written: {out_path}  ({size_mb:.1f} MB)')

On the Raspberry Pi, the bundle is loaded once at startup:

```python
import joblib
bundle = joblib.load('dt_hres_s_deployment.joblib')
model = bundle['model']
features = bundle['features']
```

Inference on a single query is immediate. No retraining, no network, no license.


## 4. Language support

The interface is built to run in more than one language. Below, Spanish and English
are provided, together with a first draft of Yucatec Maya (maya yucateco).

The Maya strings are a preliminary draft and must be reviewed and corrected by a
native speaker before any deployment. They are marked accordingly. Placing a
language the community speaks at the center of the tool is part of the project's
participatory approach, and getting it wrong undermines that intent, so this table
is a starting point for that review, not a finished translation.


In [ ]:
# NOTE: The "maya" entries are an UNVALIDATED draft.
# They must be reviewed and corrected by a native Yucatec Maya speaker
# before the instrument is used in the community.

STRINGS = {
    'title': {
        'es': 'Sistema de energia',
        'en': 'Energy system',
        'maya': 'Meyaj muuk\'',            # DRAFT - needs native review
    },
    'houses': {
        'es': 'Cuantas casas?',
        'en': 'How many houses?',
        'maya': 'Jayp\'eel naj?',          # DRAFT - needs native review
    },
    'hours': {
        'es': 'Cuantas horas al dia?',
        'en': 'How many hours per day?',
        'maya': 'Jayp\'eel oora sansamal?', # DRAFT - needs native review
    },
    'panels': {
        'es': 'Paneles solares',
        'en': 'Solar panels',
        'maya': 'Paneles solares',          # DRAFT - needs native review
    },
    'wind': {
        'es': 'Turbina de viento',
        'en': 'Wind turbine',
        'maya': 'Mola\'ay iik\'',           # DRAFT - needs native review
    },
    'battery': {
        'es': 'Bateria',
        'en': 'Battery',
        'maya': 'Bateria',                  # DRAFT - needs native review
    },
    'coverage': {
        'es': 'Cubre tu consumo',
        'en': 'Covers your demand',
        'maya': 'Ku nuupik a k\'a\'abet',   # DRAFT - needs native review
    },
    'calculate': {
        'es': 'Calcular',
        'en': 'Calculate',
        'maya': 'Xok',                      # DRAFT - needs native review
    },
}

def t(key, lang):
    """Return the string for a key in the requested language, falling back to Spanish."""
    return STRINGS.get(key, {}).get(lang, STRINGS.get(key, {}).get('es', key))

# quick preview
for lang in ('es', 'en', 'maya'):
    print(lang, '|', t('houses', lang), '|', t('calculate', lang))

## 5. Sizing logic

The device answers a practical question: given a community's demand, what system
size covers it. The function below estimates a daily demand from a small number of
inputs a community member can provide, then reports the reference system and the
share of demand it covers.

The estimate is intentionally simple and transparent. It is meant to be understood
and questioned by the person using it, not treated as a closed calculation.


In [ ]:
# Typical daily consumption per appliance, in watt-hours per day.
# These are reference values; they can be edited to match the community.
APPLIANCE_WH = {
    'light': 40,       # a few LED lights
    'fridge': 1000,    # a small refrigerator
    'pump': 1500,      # a water pump, intermittent
    'tv_radio': 200,   # television or radio
}

def estimate_demand(houses, appliances, hours):
    """Rough daily demand in kWh from simple inputs."""
    per_house = sum(APPLIANCE_WH[a] for a in appliances)
    daily_wh = per_house * houses * (hours / 8.0)  # 8h reference usage
    return daily_wh / 1000.0

def recommend_system(daily_kWh):
    """Map a daily demand to a reference PV + wind + battery size.

    This is a transparent rule of thumb for the preview. On the device it is
    replaced by the trained model's estimate plus a coverage check.
    """
    pv_kWp = round(daily_kWh / 4.5, 1)          # ~4.5 sun-hours reference
    battery_kWh = round(daily_kWh * 1.5, 1)     # 1.5 days of autonomy
    wind_kW = round(pv_kWp * 0.5, 1)            # wind complements solar
    coverage = min(0.99, 0.75 + 0.05 * (pv_kWp > 0))
    return {'pv_kWp': pv_kWp, 'wind_kW': wind_kW,
            'battery_kWh': battery_kWh, 'coverage': coverage}

demo = estimate_demand(houses=12, appliances=['light', 'fridge', 'pump'], hours=6)
print(f'Estimated daily demand: {demo:.1f} kWh')
print('Recommended system:', recommend_system(demo))

## 6. Interface preview

The device has no keyboard or touch screen. It is operated with a wheel and three
buttons. Here, the same interface is previewed with simple controls so the flow can
be reviewed before it is ported to the hardware.

On the Raspberry Pi this same logic is driven by `gpiozero`: the wheel changes a
value or moves the selection, the green button advances, amber goes back, red
restarts. The screen shows one question at a time and, at the end, the recommended
system in plain terms.


In [ ]:
LANG = 'es'  # 'es', 'en' or 'maya'

try:
    import ipywidgets as widgets
    from IPython.display import display

    lang_sel = widgets.ToggleButtons(
        options=[('Espanol', 'es'), ('English', 'en'), ('Maya (draft)', 'maya')],
        value='es', description='',
    )
    houses = widgets.IntSlider(value=12, min=1, max=50, description='')
    hours = widgets.IntSlider(value=6, min=1, max=24, description='')
    appliances = widgets.SelectMultiple(
        options=[('Luz', 'light'), ('Refrigerador', 'fridge'),
                 ('Bomba de agua', 'pump'), ('TV / Radio', 'tv_radio')],
        value=('light', 'fridge'), rows=4, description='',
    )
    out = widgets.Output()

    def refresh(_=None):
        lang = lang_sel.value
        with out:
            out.clear_output()
            d = estimate_demand(houses.value, list(appliances.value), hours.value)
            r = recommend_system(d)
            print(t('houses', lang), houses.value)
            print(t('hours', lang), hours.value)
            print('-' * 32)
            print(f"{t('panels', lang):20s} {r['pv_kWp']} kWp")
            print(f"{t('wind', lang):20s} {r['wind_kW']} kW")
            print(f"{t('battery', lang):20s} {r['battery_kWh']} kWh")
            print('-' * 32)
            print(f"{t('coverage', lang)}: {r['coverage']*100:.0f} %")

    for w in (lang_sel, houses, hours, appliances):
        w.observe(refresh, names='value')

    print('DT-HRES-S interface preview')
    display(lang_sel, houses, hours, appliances, out)
    refresh()
except ImportError:
    # Fallback for a plain Python environment (closer to the device)
    d = estimate_demand(12, ['light', 'fridge', 'pump'], 6)
    r = recommend_system(d)
    print('DT-HRES-S')
    print(f"{t('panels', LANG)}: {r['pv_kWp']} kWp")
    print(f"{t('wind', LANG)}: {r['wind_kW']} kW")
    print(f"{t('battery', LANG)}: {r['battery_kWh']} kWh")
    print(f"{t('coverage', LANG)}: {r['coverage']*100:.0f} %")

## 7. Neural network tuning over leave-one-city-out

This section is the pending task. It sets up a fair hyperparameter search for the
neural network, evaluated with the same held-out-city folds used for every other
model, and reports the four models side by side.

The held-out city stands in for Ixil, the site we cannot sensor yet. A model that
scores well when a whole city is withheld is a model we can trust to estimate the
power a hybrid system will need at a location with no local data.


In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from itertools import product

# Candidate configurations for the neural network.
# Kept small so it runs in Colab; widen once the loop is trusted.
NN_GRID = {
    'hidden_layer_sizes': [(64,), (128,), (128, 64), (256, 128)],
    'alpha': [1e-4, 1e-3, 1e-2],          # L2 regularization
    'learning_rate_init': [1e-3, 5e-3],
}

def nn_configs(grid):
    keys = list(grid.keys())
    for combo in product(*(grid[k] for k in keys)):
        yield dict(zip(keys, combo))

def build_nn(cfg):
    # Scaling matters for MLPs; the tree models did not need it.
    return make_pipeline(
        StandardScaler(),
        MLPRegressor(
            max_iter=400, early_stopping=True, n_iter_no_change=15,
            random_state=42, **cfg,
        ),
    )

print(f'{sum(1 for _ in nn_configs(NN_GRID))} NN configurations to evaluate')

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_squared_error

def cv_rmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return 100.0 * rmse / np.mean(y_true)

def evaluate_loco(estimator_fn, df, features, target='p_pv_W'):
    """Leave-one-city-out evaluation. estimator_fn() returns a fresh model.

    Mirrors the folds used in ml_models.leave_one_city_out so results are
    comparable across all four algorithms.
    """
    work = ml_models.cyclical_encode(df)
    rows = []
    for held in work['city'].unique():
        tr = work[work['city'] != held]
        te = work[work['city'] == held]
        Xtr = tr[features].dropna()
        ytr = tr.loc[Xtr.index, target]
        Xte = te[features].dropna()
        yte = te.loc[Xte.index, target]
        model = estimator_fn()
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        rows.append({
            'held_out_city': held,
            'R2': r2_score(yte, pred),
            'MAPE': mean_absolute_percentage_error(yte, pred),
            'CV_RMSE_pct': cv_rmse(yte, pred),
        })
    return pd.DataFrame(rows)

# Search the NN grid; keep the configuration with the best mean held-out R2.
best_cfg, best_r2, best_table = None, -np.inf, None
for cfg in nn_configs(NN_GRID):
    table = evaluate_loco(lambda c=cfg: build_nn(c), df_all, features)
    mean_r2 = table['R2'].mean()
    if mean_r2 > best_r2:
        best_cfg, best_r2, best_table = cfg, mean_r2, table

print('Best NN configuration:', best_cfg)
print(f'Mean held-out R2: {best_r2:.3f}')
best_table.round(3)

In [ ]:
# Side-by-side comparison of all four models on the same folds.
def make_tree_baselines():
    from sklearn.tree import DecisionTreeRegressor
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.svm import SVR
    return {
        'DecisionTree': lambda: DecisionTreeRegressor(max_depth=20, min_samples_leaf=5, random_state=42),
        'RandomForest': lambda: RandomForestRegressor(n_estimators=200, max_depth=20,
                                                      min_samples_leaf=5, n_jobs=-1, random_state=42),
        'SVM': lambda: make_pipeline(StandardScaler(), SVR(C=10, gamma='scale')),
        'NeuralNetwork': lambda: build_nn(best_cfg),
    }

summary = {}
for name, fn in make_tree_baselines().items():
    table = evaluate_loco(fn, df_all, features)
    summary[name] = {'R2': table['R2'].mean(), 'CV_RMSE_pct': table['CV_RMSE_pct'].mean()}

comparison = pd.DataFrame(summary).T.sort_values('R2', ascending=False)
print('Leave-one-city-out, mean across held-out cities:')
comparison.round(3)

If Random Forest still leads this table after the neural network is properly
tuned, that is the evidence needed to fix it as the deployment model. If not, the
choice is revisited. The point is that the decision now rests on a fair comparison.


## Note for Braulio — pending task before the physical device

Braulio, there is one task that has to be closed before we move to the physical
device. The pipeline already runs end to end: physics simulation, training of the
four models, and leave-one-city-out validation. What is missing is a fair tuning of
the neural network.

Right now the comparison between the four algorithms (Decision Tree, Random Forest,
SVM, Neural Network) is not fair, because the NN is under-configured. The prototype
notebook itself flags it: the default MLP is undertrained. Until the network is
given a real chance, any statement that Random Forest wins is not defensible.

What we need from you is a real hyperparameter search over the neural network:
architecture (number and width of layers), learning rate, and regularization,
evaluated honestly with the leave-one-city-out scheme already implemented in
`ml_models.py`. Not a random split, the same city-held-out folds, so the number
reflects generalization to a site the model never saw.

Why leave-one-city-out matters here specifically: the location we ultimately care
about is Ixil, and we do not yet have the permits to place sensors there. So Ixil
is the city that is missing from the data. The held-out-city validation is our way
of estimating how the model will behave in a place it has no local data for, and a
well-tuned neural network is what lets us recreate that behavior and keep the
modeling moving while the permits are pending.

The outcome we are after is a clean, honest comparison. If, after the network is
properly tuned, Random Forest still comes out ahead on the held-out city, then we
can state with evidence that Random Forest is the model we will deploy, the one best
suited to predict the power the hybrid system will need. If the neural network
closes the gap or wins, we reconsider. Either way, the decision must rest on a fair
fight, not on an undertrained baseline.

Concretely: use the `leave_one_city_out` function in `ml_models.py` as the
evaluation loop, wrap the NN training in a hyperparameter search around it, and
report R2, MAPE and CV-RMSE per held-out city for all four models side by side.
Document the winning configuration so it can be reproduced.


---

EPICS in IEEE 2025-2026 | Tecnológico de Monterrey | Technical Arts, ITESM Student Chapter

Project lead | PhD Rasikh Tariq
